# Lab 1 · Replicate value and momentum in Asia-Pacific

**Week 1 · Portfolio foundations & factor investing** · ≈ 75 minutes

You will rebuild two of the most famous factors from their building blocks, measure them, and decide
whether they clear the t > 3 bar from Lesson 1.2.

1. Build HML (value) from six size/book-to-market portfolios
2. Build WML (momentum) from six size/prior-return portfolios
3. Annualise returns, volatility, Sharpe ratios and t-statistics
4. Measure the value–momentum correlation and the benefit of combining them
5. Test whether value is just market beta, with a CAPM regression

**Data.** Monthly returns from the Kenneth R. French Data Library (Tuck School of Business, Dartmouth), *Asia Pacific ex Japan* region: Australia, **Hong Kong**, New Zealand and Singapore, July 1990 onwards. The setup cell downloads it once and caches it in `finex_data/`.

**How this lab works.** Run the setup cell, then work top to bottom. Each exercise has a `# TODO` cell followed by an automatic check (✓ / ✗ with a hint). When every check passes, the last cell prints a **completion code**: paste it into the course page to complete the lab.

In [1]:
# FinEx Learning · lab setup. Run this cell first; it only defines helpers.
# Data: Kenneth R. French Data Library (Tuck School of Business, Dartmouth),
# Asia Pacific ex Japan series (Australia, Hong Kong, New Zealand, Singapore),
# downloaded on first use and cached next to the notebook.
import hashlib
import io
import urllib.request
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

FRENCH = "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/"
CACHE = Path("finex_data")
CACHE.mkdir(exist_ok=True)


def _lines(name):
    path = CACHE / f"{name}.csv"
    if not path.exists():
        with urllib.request.urlopen(f"{FRENCH}{name}_CSV.zip", timeout=60) as response:
            archive = zipfile.ZipFile(io.BytesIO(response.read()))
            path.write_bytes(archive.read(archive.namelist()[0]))
    return path.read_text(encoding="latin-1").splitlines()


def load_french(name, table=None, percent=True):
    """One monthly table from a French data library file, indexed by month.

    `table` is the title above the table (e.g. "Average Value Weighted Returns -- Monthly");
    None means the first monthly table. Returns are converted to decimals.
    """
    tables, title, current = {}, "", None
    for line in _lines(name):
        cells = [c.strip() for c in line.split(",")]
        if line.lstrip().startswith(","):
            current = []
            tables.setdefault(title or "first", (cells[1:], current))
            continue
        if current is not None and len(cells[0]) == 6 and cells[0].isdigit():
            current.append(cells)
            continue
        if line.strip():
            title, current = line.strip(), None
    key = table or next(k for k, (_, rows) in tables.items() if rows)
    header, rows = tables[key]
    frame = pd.DataFrame(
        [r[1:] for r in rows],
        columns=header,
        index=pd.PeriodIndex([r[0] for r in rows], freq="M"),
    ).astype(float).replace(-99.99, np.nan)
    return frame / 100 if percent else frame


def check(name, condition, hint):
    """Automatic check: passes quietly, or explains what to look at."""
    if not bool(condition):
        raise AssertionError(f"✗ {name}. Hint: {hint}")
    print(f"✓ {name}")


def completion_code(lab):
    code = hashlib.sha256(f"finex-quant-{lab}".encode()).hexdigest()[:6].upper()
    print(f"\nAll checks passed. Your completion code for the course page: FINEX-{lab.upper()}-{code}")


def annualise(returns):
    """Mean, volatility, Sharpe and t-stat of monthly excess returns, annualised."""
    r = returns.dropna()
    mean, vol = r.mean() * 12, r.std() * np.sqrt(12)
    return pd.Series({"mean": mean, "vol": vol, "sharpe": mean / vol,
                      "t_stat": r.mean() / r.std() * np.sqrt(len(r)), "months": len(r)})


print("Setup complete. Data source: Kenneth R. French Data Library · Asia Pacific ex Japan.")


Setup complete. Data source: Kenneth R. French Data Library · Asia Pacific ex Japan.


In [2]:
factors = load_french("Asia_Pacific_ex_Japan_3_Factors")          # Mkt-RF, SMB, HML, RF
wml = load_french("Asia_Pacific_ex_Japan_MOM_Factor")["WML"]          # published momentum
bm6 = load_french("Asia_Pacific_ex_Japan_6_Portfolios_ME_BE-ME", "Average Value Weighted Returns -- Monthly")
prior6 = load_french("Asia_Pacific_ex_Japan_6_Portfolios_ME_Prior_12_2", "Average Value Weighted Returns -- Monthly")

print(f"{factors.index[0]} → {factors.index[-1]} · {len(factors)} months")
bm6.tail()

1990-07 → 2026-09 · 435 months


,SMALL LoBM,ME1 BM2,SMALL HiBM,BIG LoBM,ME2 BM2,BIG HiBM
2026-05,0.0176,0.0103,-0.0029,-0.0218,0.0304,-0.0089
2026-06,-0.0751,-0.0657,-0.0680,0.0117,-0.0402,-0.0892
2026-07,-0.0279,0.0035,0.0086,0.0219,0.0637,0.1029
2026-08,0.0550,0.0722,0.0398,0.0480,0.0374,-0.0038
2026-09,-0.0550,-0.0436,-0.0341,-0.0675,-0.0391,-0.0174


## Exercise 1 · Build HML

Fama and French define value as the average of the two *value* portfolios minus the average of the two
*growth* portfolios:

$$HML = \tfrac12(\text{Small Value} + \text{Big Value}) - \tfrac12(\text{Small Growth} + \text{Big Growth})$$

In `bm6`, value is `HiBM` and growth is `LoBM`.

In [3]:
my_hml = 0.5 * (bm6["SMALL HiBM"] + bm6["BIG HiBM"]) - 0.5 * (bm6["SMALL LoBM"] + bm6["BIG LoBM"])

In [4]:
check("HML replicates the published factor",
      isinstance(my_hml, pd.Series) and my_hml.corr(factors["HML"]) > 0.99,
      "average the two HiBM columns, subtract the average of the two LoBM columns")

✓ HML replicates the published factor


## Exercise 2 · Build WML

Momentum works the same way on portfolios sorted by return from month −12 to −2 (the most recent month
is skipped because of short-term reversal): winners `HiPRIOR` minus losers `LoPRIOR`.

In [5]:
my_wml = 0.5 * (prior6["SMALL HiPRIOR"] + prior6["BIG HiPRIOR"]) - 0.5 * (prior6["SMALL LoPRIOR"] + prior6["BIG LoPRIOR"])

In [6]:
check("WML replicates the published factor",
      isinstance(my_wml, pd.Series) and my_wml.corr(wml) > 0.99,
      "same recipe as HML, with HiPRIOR (winners) and LoPRIOR (losers)")

✓ WML replicates the published factor


## Exercise 3 · Measure the premia

Use the `annualise()` helper from the setup cell to build a table with one column per factor:
`MKT` (`Mkt-RF`), `SMB`, `HML` and `WML`. Then list the factors whose t-statistic is above 3, the
data-mining-adjusted hurdle proposed by Harvey, Liu & Zhu (2016).

In [7]:
stats = pd.DataFrame({
    "MKT": annualise(factors["Mkt-RF"]),
    "SMB": annualise(factors["SMB"]),
    "HML": annualise(factors["HML"]),
    "WML": annualise(wml),
})
passes_hurdle = stats.columns[stats.loc["t_stat"] > 3].tolist()
stats.round(3)

,MKT,SMB,HML,WML
mean,0.076,-0.032,0.071,0.103
vol,0.196,0.098,0.105,0.144
sharpe,0.387,-0.326,0.677,0.716
t_stat,2.327,-1.965,4.078,4.291
months,435.000,435.000,435.000,431.000


In [8]:
reference = {k: annualise(v)["t_stat"] for k, v in
             {"MKT": factors["Mkt-RF"], "SMB": factors["SMB"], "HML": factors["HML"], "WML": wml}.items()}
check("Statistics table has the four factors",
      isinstance(stats, pd.DataFrame) and set(stats.columns) == set(reference),
      "columns should be MKT, SMB, HML and WML")
check("t-statistics are correct",
      all(np.isclose(stats.loc["t_stat", k], v) for k, v in reference.items()),
      "use annualise() on the monthly excess returns")
check("You found the factors that clear t > 3",
      sorted(passes_hurdle or []) == sorted(k for k, v in reference.items() if v > 3),
      "filter stats.columns on stats.loc['t_stat'] > 3")

✓ Statistics table has the four factors
✓ t-statistics are correct
✓ You found the factors that clear t > 3


## Exercise 4 · Why value and momentum belong together

Compute the correlation between HML and WML, then the Sharpe ratio of a 50/50 combination.

In [9]:
value_momentum_corr = factors["HML"].corr(wml)
combo_sharpe = annualise(0.5 * factors["HML"] + 0.5 * wml)["sharpe"]
print(f"Correlation {value_momentum_corr:.2f} · combined Sharpe {combo_sharpe:.2f}")

Correlation -0.30 · combined Sharpe 1.16


In [10]:
check("Correlation is right",
      value_momentum_corr is not None and np.isclose(value_momentum_corr, factors["HML"].corr(wml)),
      "factors['HML'].corr(wml)")
check("The combination beats both factors on its own",
      combo_sharpe is not None and combo_sharpe > max(annualise(factors["HML"])["sharpe"], annualise(wml)["sharpe"]),
      "average the two monthly series first, then annualise")

✓ Correlation is right
✓ The combination beats both factors on its own


## Exercise 5 · Is value just market beta?

Regress HML on the market (`Mkt-RF`) with Newey–West (HAC, 6 lags) standard errors. If the intercept
(alpha) stays significant, value is not simply a disguised bet on the market.

In [11]:
import statsmodels.api as sm

capm = sm.OLS(factors["HML"], sm.add_constant(factors[["Mkt-RF"]])).fit(cov_type="HAC", cov_kwds={"maxlags": 6})
alpha_t = capm.tvalues["const"]
print(capm.summary().tables[1])

                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0058      0.001      4.154      0.000       0.003       0.009
Mkt-RF         0.0193      0.053      0.366      0.714      -0.084       0.123


In [12]:
reference_t = sm.OLS(factors["HML"], sm.add_constant(factors[["Mkt-RF"]])).fit(
    cov_type="HAC", cov_kwds={"maxlags": 6}).tvalues["const"]
check("Alpha t-statistic is right", alpha_t is not None and np.isclose(alpha_t, reference_t),
      "sm.OLS(y, sm.add_constant(X)).fit(cov_type='HAC', cov_kwds={'maxlags': 6})")
completion_code("lab1")

✓ Alpha t-statistic is right

All checks passed. Your completion code for the course page: FINEX-LAB1-BF739B


## Reflect (bring to Tuesday's session)

* Which factors clear t > 3 in Asia-Pacific, and does that match what you expected?
* Value and momentum are negatively correlated. What does that imply for a portfolio that holds both?
* SMB has been *negative* in this region. What explanations would you test before dropping it?